In [1]:
import logging

import numpy as np
import torch

import torch.nn as nn
import torch.nn.functional as F

# Time encoding function
Time encoding function as implemented in Xu et al. (2020):
$$\Phi_d(t) = \cos(\omega_d \cdot t + b)$$
* $t$: time span between 2 timestamps $t_1$ and $t_2$
* $d$: time embedding dimension
* $\omega_d$: frequencies (controls how fase each cosine dimension oscillates over time) (trainable parameter)
* $b$: phase shifts (trainable parameter)

In [2]:
# time encoding function
class TimeEncoding(torch.nn.Module):
    def __init__(self, expand_dim, factor=5):
        super(TimeEncoding, self).__init__()
        time_dim = expand_dim
        self.factor = factor
        self.basis_freq = torch.nn.Parameter((torch.from_numpy(1 / 10 ** np.linspace(0,9,time_dim))).float())
        self.phase = torch.nn.Parameter()
    def forward(self, ts):
        # ts: [N, L]
        batch_size = ts.size(0)
        seq_len = ts.size(1)

        ts = ts.view(batch_size, seq_len, 1)   # [N, L, 1]
        map_ts = ts * self.basis_freq.view(1, 1, -1)    # [N, L, time_dim]
        map_ts += self.phase.view(1, 1, -1)
        harmonic = torch.cos(map_ts)           # cosine transform

        return harmonic

# Alternatives to Time encoding

In [3]:
class PosEncoding(torch.nn.Module):
    def __init__(self, expand_dim, seq_len):
        super().__init__()
        self.pos_embeddings = nn.Embedding(num_embeddings=seq_len, embedding_dim=expand_dim)
    def forward(self, ts):
        # ts [N, L]
        order = ts.argsort()
        ts_emb = self.pos_embeddings(order)
        return ts_emb

In [4]:
class EmptyEncoding(torch.nn.Module):
    def __init__(self, expand_dim):
        super().__init__()
        self.expand_dim = expand_dim
        
    def forward(self, ts):
        out = torch.zeros_like(ts).float()
        out = torch.unsqueeze(out, dim=-1)
        out = out.expand(out.shape[0], out.shape[1], self.expand_dim)
        return out

# Attention Layer

Scaled dot-product attention:
$$Attn(Q,K,V) = \text{softmax}(\frac{QK^\top}{\sqrt{d}})V$$

In [5]:
class ScaledDotProductAttention(torch.nn.Module):
    """
    Scaled Dot-product Attention
    """
    def __init__(self, temperature, attn_dropout=0.1):
        super().__init__()
        self.temperature = temperature
        self.dropout = torch.nn.Dropout(attn_dropout)
        self.softmax = torch.nn.Softmax(dim=2)

    def forward(self, q, k, v, mask=None):
        attn = torch.bmm(q, k.transpose(1, 2))     # batch matrix multiplication of q and k
        attn = attn / self.temperature
        if mask is not None:
            attn = attn.masked_fill(mask, -1e10)
        attn = self.softmax(attn)                  # [n*b, l_q, l_k]
        attn = self.dropout(attn)
        output = torch.bmm(attn, v)
        return output, attn

In [6]:
class MultiHeadAttention(torch.nn.Module):
    """
    Multi-head attention module
    """
    def __init__(self, n_head, d_model, d_k, d_v, dropout=0.1):
        super().__init__()

        self.n_head = n_head
        self.d_k = d_k
        self.d_v = d_v
        
        self.w_qs = nn.Linear(d_model, n_head * d_k, bias=False)
        self.w_ks = nn.Linear(d_model, n_head * d_k, bias=False)
        self.w_vs = nn.Linear(d_model, n_head * d_v, bias=False)
        
        nn.init.normal_(self.w_qs.weight, mean=0, std=np.sqrt(2.0/(d_model + d_k)))
        nn.init.normal_(self.w_ks.weight, mean=0, std=np.sqrt(2.0/(d_model + d_k)))
        nn.init.normal_(self.w_vs.weight, mean=0, std=np.sqrt(2.0/(d_model + d_v)))

        self.attention = ScaledDotProductAttention(temperature=np.power(d_k, 0.5), attn_dropout=dropout)
        self.layer_norm = nn.LayerNorm(d_model)

        self.fc = nn.Linear(n_head * d_v, d_model)
        nn.init.xavier_normal_(self.fc.weight)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        d_k, d_v, n_head = self.d_k, self.d_v, self.n_head
        
        sz_b, len_q, _ = q.size()
        sz_b, len_k, _ = k.size()
        sz_b, len_v, _ = v.size()
        
        residual = q
        # passing input through linear layers
        # linear projections & splitting -> higher dimension input (n_head * d_)
        q = self.w_qs(q).view(sz_b, len_q, n_head, d_k)
        k = self.w_ks(k).view(sz_b, len_k, n_head, d_k)
        v = self.w_vs(v).view(sz_b, len_v, n_head, d_v)
        # transposing for batch matrix multiplication
        q = q.permute(2, 0, 1, 3).contiguous().view(-1, len_q, d_k) # (n*b) x lq x dk
        k = k.permute(2, 0, 1, 3).contiguous().view(-1, len_k, d_k) # (n*b) x lk x dk
        v = v.permute(2, 0, 1, 3).contiguous().view(-1, len_v, d_v) # (n*b) x lv x dv

        # parallel attention
        mask = mask.repeat(n_head, 1, 1) # (n*b) x .. x ..
        output, attn = self.attention(q, k, v, mask=mask)    # calculates the scores for all heads at once

        # concatenation
        # reverses the splitting process
        output = output.view(n_head, sz_b, len_q, d_v)
        output = output.permute(1, 2, 0, 3).contiguous().view(sz_b, len_q, -1) # b x lq x (n*dv)

        # final projection and residual connection
        output = self.dropout(self.fc(output))
        output = self.layer_norm(output + residual)
        #output = self.layer_norm(output)
        
        return output, attn

# Aggregation layers

In [7]:
class MergeLayer(torch.nn.Module):
    def __init__(self, dim1, dim2, dim3, dim4):
        super().__init__()
        #self.layer_norm = torch.nn.LayerNorm(dim1 + dim2)
        self.fc1 = torch.nn.Linear(dim1 + dim2, dim3)
        self.fc2 = torch.nn.Linear(dim3, dim4)
        self.act = torch.nn.ReLU()

        torch.nn.init.xavier_normal_(self.fc1.weight)
        torch.nn.init.xavier_normal_(self.fc2.weight)
        
    def forward(self, x1, x2):
        # feature concat
        x = torch.cat([x1, x2], dim=1)
        # passes combined vector through a linear layer (fc1)
        # that maps input from dim1+dim2 to hidden dim (dim3)
        # then applies ReLU
        h = self.act(self.fc1(x))
        # passes hidden representation h through second linear layer (fc2)
        # to project it to the final output dim (dim4) -> fused representation
        return self.fc2(h)

In [8]:
class MeanPool(torch.nn.Module):
    def __init__(self, feat_dim, edge_dim):
        super(MeanPool, self).__init__()
        self.edge_dim = edge_dim
        self.feat_dim = feat_dim
        self.act = torch.nn.ReLU()
        self.merger = MergeLayer(edge_dim + feat_dim, feat_dim, feat_dim, feat_dim)
        
    def forward(self, src, src_t, seq, seq_t, seq_e, mask):
        # seq [B, N, D]
        # mask [B, N]
        src_x = src
        seq_x = torch.cat([seq, seq_e], dim=2)    # [B, N, De + D]
        hn = seq_x.mean(dim=1)                    # [B, De+D]
        output = self.merger(hn, src_x)
        return output, None

In [9]:
class LSTMPool(torch.nn.Module):
    def __init__(self, feat_dim, edge_dim, time_dim):
        super(LSTMPool, self).__init__()
        self.feat_dim = feat_dim
        self.time_dim = time_dim
        self.edge_dim = edge_dim

        self.att_dim = feat_dim + edge_dim + time_dim
        self.act = torch.nn.ReLU()
        self.lstm = torch.nn.LSTM(input_size=self.att_dim,
                                 hidden_size=self.feat_dim,
                                 num_layers=1,
                                 batch_first=True)

        self.merger = MergeLayer(feat_dim, feat_dim, feat_dim, feat_dim)

    def forward(self, src, src_t, seq, seq_t, seq_e, mask):
        # seq [B, N, D]
        # mask [B, N]
        seq_x = torch.cat([seq, seq_e, seq_t], dim=2)
        _, (hn, _) = self.lstm(seq_x)
        hn = hn[-1, :, :]    # hn.squeeze(dim=0)
        out = self.merger.forward(hn, src)
        return output, None

# Attention based temporal layers

In [10]:
class AttnModel(torch.nn.Module):
    """Attention based temporal layer"""
    def __init__(self, feat_dim, edge_dim, time_dim,
                attn_mode='prod', n_head=2, drop_out=0.1):
        """
        args:
          feat_dim: dim for the node features
          edge_dim: dim for the temporal edge features
          time_dim: dim for the time encoding
          attn_mode: choose from 'prod' and 'map'
          n_head: number of heads in attention
          drop_out: probability of dropping a neural.
        """
        super().__init__()
        self.feat_dim = feat_dim
        self.time_dim = time_dim

        self.edge_in_dim = (feat_dim + edge_dim + time_dim)
        self.model_dim = self.edge_in_dim
        self.merger = MergeLayer(self.model_dim, feat_dim, feat_dim, feat_dim)

        # checks if model dim is divisible by attention heads
        assert self.model_dim % n_head == 0
        self.attn_mode = attn_mode
        # multihead attention using scaled dot product attn
        if attn_mode == 'prod':
            self.multi_head_target = MultiHeadAttention(
                n_head,
                d_model=self.model_dim,
                d_k=self.model_dim // n_head,
                d_v=self.model_dim // n_head,
                dropout=drop_out
            )
            self.logger.info("Using scaled prod attention")
        else:
            raise ValueError('attn_mode can only be prod')

    def forward(self, src, src_t, seq, seq_t, seq_e, mask):
        """"Attention based temporal attention forward pass
        args:
          src: float Tensor of shape [B, D]
          src_t: float Tensor of shape [B, Dt], Dt == D
          seq: float Tensor of shape [B, N, D]
          seq_t: float Tensor of shape [B, N, Dt]
          seq_e: float Tensor of shape [B, N, De], De == D
          mask: boolean Tensor of shape [B, N], where the true value indicate a null value in the sequence.

        returns:
          output, weight

          output: float Tensor of shape [B, D]
          weight: float Tensor of shape [B, N]
        """

        src_ext = torch.unsqueeze(src, dim=1)    # src [B, 1, D]
        src_e_ph = torch.zeros_like(src_ext)
        q = torch.cat([src_ext, src_e_ph, src_t], dim=2)    # [B, 1, D+De+Dt] -> [B, 1, D]
        k = torch.cat([seq, seq_e, seq_t], dim=2)           # [B, 1, D+De+Dt] -> [B, 1, D]

        mask = torch.unsqueeze(mask, dim=2)     # mask [B, N, 1]
        mask = mask.permute([0, 2, 1])          # mask [B, 1, N]

        # target attention
        output, attn = self.multi_head_target(q=q, k=k, v=k, mask=mask)  # output: [B, 1, D + Dt], attn: [B, 1, N]
        output = output.squeeze()
        attn = attn.squeeze()
        
        output = self.merger(output, src)
        return output, attn

# TGAT model (with different config options)

In [11]:
class TGAN(torch.nn.Module):
    def __init__(self, ngh_finder, n_feat, e_feat,
                attn_mode='prod', use_time='time', agg_method='attn',
                node_dim=None, time_dim=None, num_layers=3, n_head=4, null_idx=0,
                num_heads=1, drop_out=0.1, seq_len=None):
        super(TGAN, self).__init__()

        self.num_layers = num_layers
        self.ngh_finder = ngh_finder
        self.null_idx = null_idx
        self.logger = logging.getLogger(__name__)
        self.n_feat_th = torch.nn.Parameter(torch.from_numpy(n_feat.astype(np.float32)))
        self.e_feat_th = torch.nn.Parameter(torch.from_numpy(e_feat.astype(np.float32)))
        self.edge_raw_embed = torch.nn.Embedding.from_pretrained(self.e_feat_th, padding_idx=0, freeze=True)
        self.node_raw_embed = torch.nn.Embedding.from_pretrained(self.n_feat_th, padding_idx=0, freeze=True)

        self.feat_dim = self.n_feat_th.shape[1]
        self.n_feat_dim = self.feat_dim
        self.model_dim = self.feat_dim

        self.use_time = use_time
        self.merge_layer = MergeLayer(self.feat_dim, self.feat_dim, self.feat_dim, self.feat_dim)

        # aggregation layer
        if agg_method == 'attn':
            self.logger.info('Aggregation uses attention model')
            self.attn_model_list = torch.nn.ModuleList([
                AttnModel(self.feat_dim,
                         self.feat_dim,
                         attn_mode=attn_mode,
                         n_head=n_head,
                         drop_out=drop_out) for _ in range(num_layers)
            ])

        elif agg_method == 'lstm':
            self.logger.info("Aggregation uses LSTM model")
            self.attn_model_list = torch.nn.ModuleList([
                LSTMPool(self.feat_dim, self.feat_dim, self.feat_dim) for _ in range(num_layers)
            ])
            
        elif agg_method == 'mean':
            self.logger.info("Aggregation uses constant mean model")
            self.attn_model_list = torch.nn.ModuleList([
                MeanPool(self.feat_dim, self.feat_dim) for _ in range(num_layers)
            ])

        else:
            raise ValueError('invalid agg_method value, use attn, lstm, or mean')

        # temporal encoding options
        if use_time == 'time':
            self.logger.info("Using time encoding")
            self.time_encoder = TimeEncoding(expand_dim=self.n_feat_th.shape[1])
        elif use_time == 'pos':
            assert (seq_len is not None)
            self.logger.info("Using positional encoding")
            self.time_encoder = PosEncoding(expand_dim=self.n_feat_th.shape[1], seq_len=seq_len)
        elif use_time == 'empty':
            self.logger.info("Using empty encoding")
            self.time_encoder = EmptyEncoding(expand_dim=self.n_feat_th.shape[1])
        else: 
            raise ValueError('invalid time option!')

        self.affinity_score = MergeLayer(self.feat_dim, self.feat_dim, self.feat_dim, 1)    # torch.nn.Bilinear(self.feat_dim, self.feat_dim, 1, bias=True)

    def forward(self, src_idx_1, target_idx_1, cut_time_1, num_neighbors=20):
        src_embed = self.tem_conv(src_idx_1, cut_time_1, self.num_layers, num_neighbors)
        target_embed = self.tem_conv(target_idx_1, cut_time_1, self.num_layers, num_neighbors)
        score = self.affinity_score(src_embed, target_embed).squeeze(dim=-1)
        return score

    def contrast(self, src_idx_1, target_idx_1, background_idx_1, cut_time_1, num_neighbors=20):
        src_embed = self.tem_conv(src_idx_1, cut_time_1, self.num_layers, num_neighbors)
        target_embed = self.tem_conv(target_idx_1, cut_time_1, self.num_layers, num_neighbors)
        background_embed = self.tem_conv(background_idx_1, cut_time_1, self.num_layers, num_neighbors)
        pos_score = self.affinity_score(src_embed, target_embed).squeeze(dim=-1)
        neg_score = self.affinity_score(src_embed, background_embed).squeeze(dim=-1)
        return pos_score.sigmoid(), neg_score.sigmoid()

    def tem_conv(self, src_idx_1, cut_time_1, curr_layers, num_neighbors=20):
        assert curr_layers >= 0

        device = self.n_feat_th.device
        batch_size = len(src_idx_1)

        src_node_batch_th = torch.from_numpy(src_idx_1).long().to(device)
        cut_time_1_th = torch.from_numpy(cut_time_1).float().to(device)
        cut_time_1_th = torch.unsqueeze(cut_time_1_th, dim=1)
        
        # query node always has the start time -> time span == 0
        src_node_t_embed = self.time_encoder(torch.zeros_like(cut_time_l_th))
        src_node_feat = self.node_raw_embed(src_node_batch_th)

        if curr_layers == 0:
            return src_node_feat
        else:
            src_node_conv_feat = self.tem_conv(src_idx_1,
                                              cut_time_1,
                                              curr_layers=curr_layers-1, 
                                              num_neighbors=num_neighbors)
            
            src_ngh_node_batch, src_ngh_eidx_batch, src_ngh_t_batch = self.ngh_finder.get_temporal_neighbor(
                src_idx_1, cut_time_1, num_neighbors=num_neighbors
            )

            src_ngh_node_batch_th = torch.from_numpy(src_ngh_node_batch).long().to(device)
            src_ngh_eidx_batch = torch.from_numpy(src_ngh_eidx_batch).long().to(device)

            src_ngh_t_batch_delta = cut_time_1[:, np.newaxis] - src_ngh_t_batch
            src_ngh_t_batch_th = torch.from_numpy(src_ngh_t_batch_delta).float().to(device)

            # get previous layer's node features
            src_ngh_node_batch_flat = src_ngh_node_batch.flatten()    # reshape(batch_size, -1)
            src_ngh_t_batch_flat = src_ngh_t_batch.flatten()          # reshape(batch_size, -1)
            src_ngh_node_conv_feat = self.tem_conv(src_ngh_node_batch_flat,
                                                  src_ngh_t_batch_flat,
                                                  curr_layers=curr_layers-1,
                                                  num_neighbors=num_neighbors)
            src_ngh_feat = src_ngh_node_conv_feat.view(batch_size, num_neighbors, -1)

            # get edge time features and node features
            src_ngh_t_embed = self.time_encoder(src_ngh_t_batch_th)
            src_ngn_edge_feat = self.edge_raw_embed(src_ngh_eidx_batch)

            # attn aggregation
            mask = src_ngh_node_batch_th == 0
            attn_m = self.attn_model_list[curr_layers-1]
            local, weight = attn_m(src_node_conv_feat,
                                  src_node_t_embed,
                                  src_ngh_feat,
                                  src_ngh_t_embed,
                                  src_ngn_edge_feat,
                                  mask)
            return local